# Cued Detection with Geographic ROIs

**Source**: `grdl/example/shapes/cued_detection_overlay.py`

## Learning Objectives

- Define geographic regions of interest (Circle, Ellipse, GeoPolygon)
- Run CFAR detection only inside ROI boundaries (cued detection)
- Overlay shapes and detections on imagery
- Understand when cueing reduces false alarms

## Theory: Cued Detection

**Problem**: Running blind CFAR over a full image produces false alarms in irrelevant regions (e.g., water, urban areas).

**Solution**: Geographic cueing — run the detector only inside a region of interest defined in lat/lon coordinates.

**GRDL workflow**:
1. Define ROI as a geographic shape (Circle, Ellipse, GeoPolygon)
2. Rasterize ROI to pixel mask via geolocation
3. Run detector only on masked pixels → `cued_detect(detector, image, roi, geo)`

**Benefits**:
- Reduced false alarms (ignore irrelevant regions)
- Faster processing (skip irrelevant pixels)
- Prior knowledge integration (e.g., "targets only in this field")

## Data Setup

This example uses a **synthetic scene** — no external data required. The scene contains:
- Target A: bright target at (80, 80) — **inside ROI**
- Target B: bright target at (380, 380) — **outside ROI**
- Background: Gaussian noise (mean=10, std=1)

The detector should find Target A but ignore Target B.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass  # Not running in IPython

# Validate GRDL version
import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rasterio.transform import Affine

from grdl.geolocation.eo.affine import AffineGeolocation
from grdl.image_processing.detection.cfar import CACFARDetector
from grdl.shapes import Circle, cued_detect, overlay_shape

In [ ]:
# Generate synthetic scene
def make_scene(shape=(512, 512)):
    rng = np.random.default_rng(2026)
    img = rng.normal(10.0, 1.0, shape).astype(np.float64)
    # Target A: inside ROI (will be detected)
    img[80:90, 80:90] = 200.0
    # Target B: outside ROI (will be ignored)
    img[380:390, 380:390] = 200.0
    return img

image = make_scene()
print(f"Scene shape: {image.shape}")
print(f"Scene statistics: mean={image.mean():.2f}, std={image.std():.2f}")
print(f"Target A (inside ROI): pixels [80:90, 80:90] = {image[85, 85]:.1f}")
print(f"Target B (outside ROI): pixels [380:390, 380:390] = {image[385, 385]:.1f}")

In [ ]:
# Build geolocation (affine transform: 1m/pixel GSD, WGS84)
transform = Affine(1e-3, 0.0, -118.2, 0.0, -1e-3, 34.1)
geo = AffineGeolocation(transform, image.shape, 'EPSG:4326')

# Verify corner coordinates
corners = [
    (0, 0),
    (0, image.shape[1] - 1),
    (image.shape[0] - 1, 0),
    (image.shape[0] - 1, image.shape[1] - 1),
]
print("Image corner coordinates:")
for r, c in corners:
    lat, lon, _ = geo.image_to_latlon(r, c)
    print(f"  Pixel ({r:3d}, {c:3d}) → ({lat:.4f}°, {lon:.4f}°)")

In [ ]:
# Define circular ROI centered on Target A (radius = 15 km)
roi = Circle(
    center_lat=34.1 - 85 * 1e-3,  # 85 pixels down from top
    center_lon=-118.2 + 85 * 1e-3,  # 85 pixels right from left
    radius_m=15_000.0,
)

center_lat, center_lon = roi.center_latlon
print(f"ROI center: ({center_lat:.6f}°, {center_lon:.6f}°)")
print(f"ROI radius: {roi.radius_m:,.0f} m")

# Verify Target A/B locations relative to ROI center
lat_a, lon_a, _ = geo.image_to_latlon(85, 85)
lat_b, lon_b, _ = geo.image_to_latlon(385, 385)
print(f"\nTarget A: ({lat_a:.6f}°, {lon_a:.6f}°)")
print(f"Target B: ({lat_b:.6f}°, {lon_b:.6f}°)")

In [ ]:
# Configure CA-CFAR detector
detector = CACFARDetector(
    guard_cells=2,
    training_cells=4,
    pfa=1e-3,  # P(false alarm) = 0.001
    min_pixels=4,  # Minimum detection size
    assumption='gaussian',  # Gaussian noise model
)

print("CA-CFAR configuration:")
print(f"  Guard cells: {detector.guard_cells}")
print(f"  Training cells: {detector.training_cells}")
print(f"  PFA: {detector.pfa}")
print(f"  Min pixels: {detector.min_pixels}")

In [ ]:
# Run cued detection (detections only inside ROI)
detections = cued_detect(detector, image, roi, geo)

print(f"\n{len(detections)} detection(s) inside the cued region")
for i, d in enumerate(detections):
    if d.pixel_geometry is None:
        continue
    x, y = d.pixel_geometry.centroid.x, d.pixel_geometry.centroid.y
    lat, lon = d.geographic_geometry.centroid.y, d.geographic_geometry.centroid.x
    print(f"  Detection {i+1}: pixel ({y:.0f}, {x:.0f}) → ({lat:.6f}°, {lon:.6f}°)")

In [ ]:
# Visualize: overlay ROI and detections
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(image, cmap='gray', vmin=0, vmax=50)

# Overlay ROI circle
overlay_shape(ax, roi, geo, color='cyan', linewidth=2, label='ROI')

# Overlay detections
for d in detections:
    if d.pixel_geometry is None:
        continue
    x, y = d.pixel_geometry.centroid.x, d.pixel_geometry.centroid.y
    ax.plot(x, y, 'r+', markersize=12, markeredgewidth=2)

ax.set_title(f"Cued CA-CFAR: {len(detections)} detection(s) inside ROI", fontsize=14)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Physical Interpretation

**Expected result**: 1 detection (Target A inside ROI)

**Why Target B is ignored**:
- Target B is at pixel (385, 385) — outside the 15 km radius ROI
- Cueing restricts the detector mask to pixels inside the ROI polygon
- Target B's pixels are masked out before CFAR runs

**Key insight**: Geographic cueing reduces false alarms by encoding domain knowledge:
- "Targets only appear in open terrain, not water"
- "Search only this field, not the surrounding forest"
- "Focus on this vehicle convoy route, ignore urban areas"

**Performance**: Cueing is faster than full-image CFAR because the detector skips irrelevant pixels.

## Summary

**GRDL patterns demonstrated**:
- ✅ `grdl.shapes.Circle` — geographic circle ROI
- ✅ `grdl.shapes.cued_detect()` — ROI-masked CFAR detection
- ✅ `grdl.shapes.overlay_shape()` — matplotlib shape overlay
- ✅ `AffineGeolocation` — simple affine geolocation for synthetic scenes

**When to use cueing**:
- You have prior knowledge of target locations (e.g., AOI from intelligence)
- Full-image CFAR produces too many false alarms
- You want to reduce computation time by skipping irrelevant regions